In [6]:
import os
import re
import warnings
import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    roc_auc_score,
)
from sklearn.exceptions import ConvergenceWarning

# =========================
# 0. Config
# =========================

MODEL_NAME = "FVs_medsiglip"   # 这里只改一个模型名即可

NPZ_PATH = f"../aligned_features_drvalid/aligned_drvalid_{MODEL_NAME}.npz"
META_PATH = f"../aligned_features_drvalid/aligned_drvalid_{MODEL_NAME}_metadata.csv"

GRADE_COL = "patient_DR_Level"   # 如果你想用眼级别标签，改成 "eye_DR_Level"

N_PC = 50
N_SPLITS = 5
RANDOM_STATE = 42

# 自动剔除标准：
# disease_score = abs(Spearman(PC, DR))
# binary nuisance_score = 2 * abs(AUC_raw - 0.5)
# ordinal nuisance_score = abs(Spearman(PC, nuisance))
# remove if max_nuisance_score > disease_score
#
# 如果你之前没有最低阈值，就保持 0.0
# 如果你想避免剔除很弱的PC，可以改成 0.15 或 0.20
MIN_NUISANCE_SCORE = 0.0

BINARY_NUISANCE_COLS = [
    "eye",
    "view_no",
    "Overall quality",
]

ORDINAL_NUISANCE_COLS = [
    "Clarity",
    "Field definition",
    "Artifact",
]

OUT_DIR = "./kfold_pc_removal_one_model"
os.makedirs(OUT_DIR, exist_ok=True)


# =========================
# 1. Utility functions
# =========================

def add_eye_and_view_from_path(meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    从 image_path 中解析 eye 和 view_no。
    例如:
        xxx/5_l1.jpg -> eye = l, view_no = 1
        xxx/5_r2.jpg -> eye = r, view_no = 2
    如果原本已有 eye/view_no，则不覆盖。
    """
    meta_df = meta_df.copy()

    if "image_path" not in meta_df.columns:
        return meta_df

    path_s = meta_df["image_path"].astype(str)

    if "eye" not in meta_df.columns:
        eye = path_s.str.extract(r"_[lLrR](\d)?(?:\.|$)")[0]
        # 上面只取到了数字，所以重新抽 l/r
        eye = path_s.str.extract(r"_([lLrR])\d?(?:\.|$)")[0]
        meta_df["eye"] = eye.str.lower()

    if "view_no" not in meta_df.columns:
        view_no = path_s.str.extract(r"_[lLrR](\d)(?:\.|$)")[0]
        meta_df["view_no"] = pd.to_numeric(view_no, errors="coerce")

    return meta_df


def safe_abs_spearman(x, y):
    """
    返回:
        abs_rho, raw_rho, p_value
    PCA方向正负是任意的，所以判断强度时用 abs(rho)。
    """
    x = np.asarray(x)
    y = np.asarray(y)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 3:
        return np.nan, np.nan, np.nan

    if len(np.unique(x)) < 2 or len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan

    rho, p = spearmanr(x, y)

    if np.isnan(rho):
        return np.nan, rho, p

    return abs(rho), rho, p


def binary_auc_nuisance_strength(pc_score, factor_values):
    """
    binary nuisance:
        nuisance_score = 2 * abs(AUC_raw - 0.5)

    返回:
        strength, auc_raw, n_valid
    """
    tmp = pd.DataFrame({
        "pc": pc_score,
        "factor": factor_values,
    }).dropna()

    if len(tmp) < 3:
        return np.nan, np.nan, len(tmp)

    unique_vals = sorted(tmp["factor"].unique(), key=lambda v: str(v))

    if len(unique_vals) != 2:
        return np.nan, np.nan, len(tmp)

    mapping = {
        unique_vals[0]: 0,
        unique_vals[1]: 1,
    }

    y_bin = tmp["factor"].map(mapping).astype(int).to_numpy()
    score = tmp["pc"].to_numpy()

    if len(np.unique(y_bin)) < 2:
        return np.nan, np.nan, len(tmp)

    auc_raw = roc_auc_score(y_bin, score)
    strength = 2.0 * abs(auc_raw - 0.5)

    return strength, auc_raw, len(tmp)


def select_pcs_to_remove(
    Z_train,
    meta_train,
    grade_col,
    binary_cols,
    ordinal_cols,
    min_nuisance_score=0.0,
):
    """
    在一个fold的train set里自动选择要剔除的PC。

    Z_train: PCA后的train特征, shape = [n_train, n_pc]
    meta_train: train对应的metadata
    """
    n_pc = Z_train.shape[1]

    y_dr = pd.to_numeric(meta_train[grade_col], errors="coerce").to_numpy()

    rows = []
    remove_indices = []

    for pc_idx in range(n_pc):
        pc_name = f"PC{pc_idx + 1}"
        pc_score = Z_train[:, pc_idx]

        disease_score, disease_rho, disease_p = safe_abs_spearman(pc_score, y_dr)

        nuisance_records = []

        # binary nuisance: eye, view_no, Overall quality 等
        for col in binary_cols:
            if col not in meta_train.columns:
                continue

            strength, auc_raw, n_valid = binary_auc_nuisance_strength(
                pc_score,
                meta_train[col].to_numpy()
            )

            nuisance_records.append({
                "factor": col,
                "factor_type": "binary_auc",
                "nuisance_score": strength,
                "raw_metric": auc_raw,
                "n_valid": n_valid,
            })

        # ordinal nuisance: Clarity, Field definition, Artifact 等
        for col in ordinal_cols:
            if col not in meta_train.columns:
                continue

            y_factor = pd.to_numeric(meta_train[col], errors="coerce").to_numpy()
            strength, rho, p = safe_abs_spearman(pc_score, y_factor)

            nuisance_records.append({
                "factor": col,
                "factor_type": "spearman",
                "nuisance_score": strength,
                "raw_metric": rho,
                "n_valid": np.sum(np.isfinite(y_factor)),
            })

        nuisance_df = pd.DataFrame(nuisance_records)

        if len(nuisance_df) == 0 or nuisance_df["nuisance_score"].dropna().empty:
            max_nuisance_score = np.nan
            max_nuisance_factor = None
            max_nuisance_type = None
            max_nuisance_raw_metric = np.nan
        else:
            best_row = nuisance_df.sort_values(
                "nuisance_score",
                ascending=False,
                na_position="last"
            ).iloc[0]

            max_nuisance_score = best_row["nuisance_score"]
            max_nuisance_factor = best_row["factor"]
            max_nuisance_type = best_row["factor_type"]
            max_nuisance_raw_metric = best_row["raw_metric"]

        # remove = (
        #     np.isfinite(max_nuisance_score)
        #     and np.isfinite(disease_score)
        #     and max_nuisance_score >= min_nuisance_score
        #     and max_nuisance_score > disease_score
        # )
        STRONG_NUISANCE_TH = 0.4
        WEAK_DISEASE_TH = 0.3
        NUISANCE_MARGIN_TH = 0.15

        remove = (
            np.isfinite(max_nuisance_score)
            and np.isfinite(disease_score)
            and max_nuisance_score >= STRONG_NUISANCE_TH
            and disease_score <= WEAK_DISEASE_TH
            and (max_nuisance_score - disease_score) >= NUISANCE_MARGIN_TH
        )        

        if remove:
            remove_indices.append(pc_idx)

        rows.append({
            "PC": pc_name,
            "pc_index": pc_idx,
            "disease_score_abs_spearman": disease_score,
            "disease_spearman_rho": disease_rho,
            "disease_p_value": disease_p,
            "max_nuisance_score": max_nuisance_score,
            "max_nuisance_factor": max_nuisance_factor,
            "max_nuisance_type": max_nuisance_type,
            "max_nuisance_raw_metric": max_nuisance_raw_metric,
            "remove": remove,
        })

    selection_df = pd.DataFrame(rows)
    return remove_indices, selection_df


def evaluate_linear_svm(X_train, y_train, X_valid, y_valid, labels):
    """
    Linear SVM评估。
    这里不额外Standardize PC score，保持PCA空间的尺度。
    如果仍然收敛警告，可以继续增大 max_iter。
    """
    if X_train.shape[1] == 0:
        return {
            "qwk": np.nan,
            "accuracy": np.nan,
            "macro_f1": np.nan,
        }, None

    clf = LinearSVC(
        C=1.0,
        class_weight="balanced",
        dual=False,
        max_iter=100000,
        tol=1e-4,
        random_state=RANDOM_STATE,
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        clf.fit(X_train, y_train)

    pred = clf.predict(X_valid)

    metrics = {
        "qwk": cohen_kappa_score(y_valid, pred, labels=labels, weights="quadratic"),
        "accuracy": accuracy_score(y_valid, pred),
        "macro_f1": f1_score(y_valid, pred, labels=labels, average="macro", zero_division=0),
    }

    return metrics, pred


# =========================
# 2. Multi-model K-fold evaluation
# =========================

MODEL_LIST = [
    "FVs_medsiglip",
    "FVs_FLAIR",
    "FVs_LLM2CLIP-openai",
    "FVs_ConceptCLIP",
    "FVs_biomed",
    "FVs_LLM2CLIP-EVA",
]

# 当前采用的PC剔除标准
STRONG_NUISANCE_TH = 0.40
WEAK_DISEASE_TH = 0.3
NUISANCE_MARGIN_TH = 0.15

OUT_DIR = "./kfold_pc_removal_6models"
os.makedirs(OUT_DIR, exist_ok=True)


def run_one_model(model_name):
    print("\n" + "=" * 80)
    print(f"Running model: {model_name}")
    print("=" * 80)

    npz_path = f"../aligned_features_drvalid/aligned_drvalid_{model_name}.npz"
    meta_path = f"../aligned_features_drvalid/aligned_drvalid_{model_name}_metadata.csv"

    if not os.path.exists(npz_path):
        print(f"[SKIP] npz not found: {npz_path}")
        return None, None

    if not os.path.exists(meta_path):
        print(f"[SKIP] metadata not found: {meta_path}")
        return None, None

    data = np.load(npz_path, allow_pickle=True)
    X = data["X"]

    meta_df = pd.read_csv(meta_path)
    meta_df = add_eye_and_view_from_path(meta_df)

    if len(X) != len(meta_df):
        raise ValueError(
            f"{model_name}: X and metadata length mismatch: "
            f"X={len(X)}, meta={len(meta_df)}"
        )

    if GRADE_COL not in meta_df.columns:
        raise ValueError(f"{model_name}: {GRADE_COL} not found in metadata.")

    valid_mask = meta_df[GRADE_COL].notna().to_numpy()
    finite_x_mask = np.isfinite(X).all(axis=1)
    mask = valid_mask & finite_x_mask

    X = X[mask]
    meta_df = meta_df.loc[mask].reset_index(drop=True)

    y = pd.to_numeric(meta_df[GRADE_COL], errors="coerce").astype(int).to_numpy()
    labels = sorted(np.unique(y))

    print(f"X shape after filtering: {X.shape}")
    print("Label distribution:")
    print(pd.Series(y).value_counts().sort_index())

    class_counts = pd.Series(y).value_counts()
    actual_n_splits = min(N_SPLITS, int(class_counts.min()))

    if actual_n_splits < 2:
        print(f"[SKIP] {model_name}: Not enough samples per class.")
        return None, None

    skf = StratifiedKFold(
        n_splits=actual_n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    fold_rows = []
    selection_dfs = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        print(f"\n===== {model_name} | Fold {fold}/{actual_n_splits} =====")

        X_train_raw = X[train_idx]
        X_valid_raw = X[valid_idx]

        y_train = y[train_idx]
        y_valid = y[valid_idx]

        meta_train = meta_df.iloc[train_idx].reset_index(drop=True)

        print(f"train={len(train_idx)}, valid={len(valid_idx)}")

        # scaler只能fit train
        scaler = StandardScaler()
        X_train_std = scaler.fit_transform(X_train_raw)
        X_valid_std = scaler.transform(X_valid_raw)

        # PCA只能fit train
        n_components = min(N_PC, X_train_std.shape[0] - 1, X_train_std.shape[1])

        pca = PCA(
            n_components=n_components,
            random_state=RANDOM_STATE,
        )

        Z_train = pca.fit_transform(X_train_std)
        Z_valid = pca.transform(X_valid_std)

        # 先用原函数得到每个PC的 disease/nuisance 分数
        remove_indices_old, selection_df = select_pcs_to_remove(
            Z_train=Z_train,
            meta_train=meta_train,
            grade_col=GRADE_COL,
            binary_cols=BINARY_NUISANCE_COLS,
            ordinal_cols=ORDINAL_NUISANCE_COLS,
            min_nuisance_score=0.0,
        )

        # # 重新按当前标准覆盖 remove
        # selection_df["remove"] = (
        #     np.isfinite(selection_df["max_nuisance_score"])
        #     & np.isfinite(selection_df["disease_score_abs_spearman"])
        #     & (selection_df["max_nuisance_score"] >= STRONG_NUISANCE_TH)
        #     & (selection_df["disease_score_abs_spearman"] <= WEAK_DISEASE_TH)
        #     & (
        #         selection_df["max_nuisance_score"]
        #         - selection_df["disease_score_abs_spearman"]
        #         >= NUISANCE_MARGIN_TH
        #     )
        # )

        # remove_indices = (
        #     selection_df.loc[selection_df["remove"], "pc_index"]
        #     .astype(int)
        #     .tolist()
        # )
        # =========================
        # Adaptive top-K removal rule
        # =========================

        TOP_K_REMOVE = 5              # 每个fold最多剔除几个PC，可以试 3 / 5
        MIN_NUISANCE_SCORE = 0.15     # 至少要有一定非病变相关性
        MIN_MARGIN = 0.00             # nuisance_score - disease_score 至少大于多少
        PROTECT_DISEASE_SCORE = 0.60  # 保护强DR主成分，避免误删PC1这种病变轴

        selection_df["nuisance_minus_disease"] = (
            selection_df["max_nuisance_score"]
            - selection_df["disease_score_abs_spearman"]
        )

        # 候选条件：
        # 1. 非病变分数有效
        # 2. 病变分数有效
        # 3. 非病变分数至少达到最低要求
        # 4. 非病变分数高于病变分数
        # 5. 不是非常强的DR主成分
        candidate_df = selection_df[
            np.isfinite(selection_df["max_nuisance_score"])
            & np.isfinite(selection_df["disease_score_abs_spearman"])
            & (selection_df["max_nuisance_score"] >= MIN_NUISANCE_SCORE)
            & (selection_df["nuisance_minus_disease"] >= MIN_MARGIN)
            & (selection_df["disease_score_abs_spearman"] <= PROTECT_DISEASE_SCORE)
        ].copy()

        candidate_df = candidate_df.sort_values(
            ["nuisance_minus_disease", "max_nuisance_score"],
            ascending=False
        )

        remove_indices = (
            candidate_df
            .head(TOP_K_REMOVE)["pc_index"]
            .astype(int)
            .tolist()
        )

        selection_df["remove"] = selection_df["pc_index"].isin(remove_indices)

        keep_indices = [i for i in range(n_components) if i not in remove_indices]
        removed_pcs = [f"PC{i + 1}" for i in remove_indices]

        print(f"Removed PCs: {removed_pcs if removed_pcs else 'None'}")

        selection_df.insert(0, "fold", fold)
        selection_df.insert(1, "model_name", model_name)
        selection_dfs.append(selection_df)

        # baseline: 全PC
        raw_metrics, raw_pred = evaluate_linear_svm(
            Z_train,
            y_train,
            Z_valid,
            y_valid,
            labels=labels,
        )

        # cleaned: 剔除PC后
        clean_metrics, clean_pred = evaluate_linear_svm(
            Z_train[:, keep_indices],
            y_train,
            Z_valid[:, keep_indices],
            y_valid,
            labels=labels,
        )

        row = {
            "model_name": model_name,
            "fold": fold,
            "n_train": len(train_idx),
            "n_valid": len(valid_idx),
            "n_pc_used": n_components,
            "n_removed": len(remove_indices),
            "removed_pcs": ",".join(removed_pcs),

            "raw_qwk": raw_metrics["qwk"],
            "raw_accuracy": raw_metrics["accuracy"],
            "raw_macro_f1": raw_metrics["macro_f1"],

            "clean_qwk": clean_metrics["qwk"],
            "clean_accuracy": clean_metrics["accuracy"],
            "clean_macro_f1": clean_metrics["macro_f1"],
        }

        row["delta_qwk"] = row["clean_qwk"] - row["raw_qwk"]
        row["delta_accuracy"] = row["clean_accuracy"] - row["raw_accuracy"]
        row["delta_macro_f1"] = row["clean_macro_f1"] - row["raw_macro_f1"]

        fold_rows.append(row)

    fold_results_df = pd.DataFrame(fold_rows)
    selection_all_df = pd.concat(selection_dfs, ignore_index=True)

    # 保存单模型结果
    fold_results_path = os.path.join(OUT_DIR, f"{model_name}_kfold_performance.csv")
    selection_path = os.path.join(OUT_DIR, f"{model_name}_pc_selection_detail.csv")

    fold_results_df.to_csv(fold_results_path, index=False, encoding="utf-8-sig")
    selection_all_df.to_csv(selection_path, index=False, encoding="utf-8-sig")

    print(f"\nSaved fold results to: {fold_results_path}")
    print(f"Saved PC selection detail to: {selection_path}")

    return fold_results_df, selection_all_df


# =========================
# 3. Run all selected models
# =========================

all_fold_results = []
all_selection_results = []

for model_name in MODEL_LIST:
    fold_df, selection_df = run_one_model(model_name)

    if fold_df is not None:
        all_fold_results.append(fold_df)

    if selection_df is not None:
        all_selection_results.append(selection_df)

all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
all_selection_results_df = pd.concat(all_selection_results, ignore_index=True)

all_fold_results_path = os.path.join(OUT_DIR, "ALL_6models_kfold_performance.csv")
all_selection_results_path = os.path.join(OUT_DIR, "ALL_6models_pc_selection_detail.csv")

all_fold_results_df.to_csv(all_fold_results_path, index=False, encoding="utf-8-sig")
all_selection_results_df.to_csv(all_selection_results_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("All models finished.")
print(f"Saved: {all_fold_results_path}")
print(f"Saved: {all_selection_results_path}")
print("=" * 80)


# =========================
# 4. Summary by model
# =========================

summary_by_model = (
    all_fold_results_df
    .groupby("model_name")
    .agg(
        raw_qwk_mean=("raw_qwk", "mean"),
        raw_qwk_std=("raw_qwk", "std"),
        clean_qwk_mean=("clean_qwk", "mean"),
        clean_qwk_std=("clean_qwk", "std"),
        delta_qwk_mean=("delta_qwk", "mean"),
        delta_qwk_std=("delta_qwk", "std"),

        raw_acc_mean=("raw_accuracy", "mean"),
        clean_acc_mean=("clean_accuracy", "mean"),
        delta_acc_mean=("delta_accuracy", "mean"),

        raw_macro_f1_mean=("raw_macro_f1", "mean"),
        clean_macro_f1_mean=("clean_macro_f1", "mean"),
        delta_macro_f1_mean=("delta_macro_f1", "mean"),

        n_removed_mean=("n_removed", "mean"),
        n_removed_std=("n_removed", "std"),
    )
    .reset_index()
    .sort_values("raw_qwk_mean", ascending=False)
)

display(summary_by_model)

summary_path = os.path.join(OUT_DIR, "ALL_6models_summary_by_model.csv")
summary_by_model.to_csv(summary_path, index=False, encoding="utf-8-sig")

print(f"Saved summary to: {summary_path}")


Running model: FVs_medsiglip
X shape after filtering: (1189, 1152)
Label distribution:
0    353
1    238
2    238
3    240
4    120
Name: count, dtype: int64

===== FVs_medsiglip | Fold 1/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC5', 'PC3', 'PC6', 'PC7']

===== FVs_medsiglip | Fold 2/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC5', 'PC3', 'PC7', 'PC17']

===== FVs_medsiglip | Fold 3/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC5', 'PC7', 'PC3', 'PC4']

===== FVs_medsiglip | Fold 4/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC5', 'PC3', 'PC7', 'PC6']

===== FVs_medsiglip | Fold 5/5 =====
train=952, valid=237
Removed PCs: ['PC2', 'PC5', 'PC3', 'PC6', 'PC7']

Saved fold results to: ./kfold_pc_removal_6models\FVs_medsiglip_kfold_performance.csv
Saved PC selection detail to: ./kfold_pc_removal_6models\FVs_medsiglip_pc_selection_detail.csv

Running model: FVs_FLAIR
X shape after filtering: (1189, 512)
Label distribution:
0    353
1    238
2    238
3    240

,model_name,raw_qwk_mean,raw_qwk_std,clean_qwk_mean,clean_qwk_std,delta_qwk_mean,delta_qwk_std,raw_acc_mean,clean_acc_mean,delta_acc_mean,raw_macro_f1_mean,clean_macro_f1_mean,delta_macro_f1_mean,n_removed_mean,n_removed_std
1,FVs_FLAIR,0.872296,0.017474,0.852973,0.010063,-0.019323,0.019891,0.754409,0.735918,-0.018491,0.715518,0.699440,-0.016078,5.0,0.0
5,FVs_medsiglip,0.865744,0.028312,0.847283,0.038883,-0.018461,0.021774,0.767014,0.744318,-0.022696,0.734629,0.711554,-0.023075,5.0,0.0
3,FVs_LLM2CLIP-openai,0.817659,0.021790,0.815606,0.023680,-0.002053,0.028728,0.703968,0.689654,-0.014314,0.665008,0.646215,-0.018793,5.0,0.0
0,FVs_ConceptCLIP,0.799611,0.031743,0.746223,0.063921,-0.053388,0.049514,0.677850,0.635826,-0.042024,0.632207,0.585738,-0.046469,5.0,0.0
4,FVs_biomed,0.749705,0.026839,0.734236,0.046908,-0.015469,0.031585,0.592104,0.576123,-0.015981,0.539995,0.526444,-0.013551,5.0,0.0
2,FVs_LLM2CLIP-EVA,0.613238,0.033534,0.609539,0.025972,-0.003699,0.012633,0.529873,0.534057,0.004184,0.470699,0.474145,0.003445,5.0,0.0


Saved summary to: ./kfold_pc_removal_6models\ALL_6models_summary_by_model.csv


In [4]:
removed_summary_by_model = (
    all_selection_results_df[all_selection_results_df["remove"]]
    .groupby(["model_name", "PC"])
    .agg(
        removed_count=("remove", "count"),
        mean_disease_score=("disease_score_abs_spearman", "mean"),
        mean_nuisance_score=("max_nuisance_score", "mean"),
        main_nuisance_factor=(
            "max_nuisance_factor",
            lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else None
        ),
    )
    .reset_index()
    .sort_values(
        ["model_name", "removed_count", "mean_nuisance_score"],
        ascending=[True, False, False]
    )
)

display(removed_summary_by_model)

removed_summary_path = os.path.join(
    OUT_DIR,
    "ALL_6models_removed_pc_summary.csv"
)

removed_summary_by_model.to_csv(
    removed_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved removed PC summary to: {removed_summary_path}")

,model_name,PC,removed_count,mean_disease_score,mean_nuisance_score,main_nuisance_factor
0,FVs_ConceptCLIP,PC2,5,0.131272,0.894974,eye
1,FVs_ConceptCLIP,PC3,5,0.105313,0.597109,Clarity
2,FVs_ConceptCLIP,PC5,3,0.024678,0.504748,view_no
4,FVs_LLM2CLIP-openai,PC5,5,0.020355,0.606462,view_no
3,FVs_LLM2CLIP-openai,PC2,5,0.085176,0.474425,Clarity
5,FVs_biomed,PC4,1,0.184424,0.402277,view_no
6,FVs_medsiglip,PC2,5,0.021710,0.922959,eye
8,FVs_medsiglip,PC5,5,0.071505,0.618597,view_no
7,FVs_medsiglip,PC3,5,0.125241,0.466644,Clarity
10,FVs_medsiglip,PC7,1,0.078864,0.449637,view_no


Saved removed PC summary to: ./kfold_pc_removal_6models\ALL_6models_removed_pc_summary.csv
